# Exercise: Building a Dashboard with Quarto

In this exercise you will build an **interactive dashboard** from scratch using Quarto's `format: dashboard` — entirely from a Jupyter notebook (`.ipynb` file).

We will use the **Gapminder** dataset and **Plotly Express** to create a multi-row, multi-page dashboard with animated charts.

## Step 0: Install Quarto (Linux / micromamba)

If you don't have Quarto installed yet, the easiest way on Linux is via **micromamba** 

### Create an environment and install Quarto

```bash
# Create a new environment with Quarto and the Python packages we need
micromamba create -n dashboard -c conda-forge quarto plotly pandas jupyter -y

# Activate the environment
micromamba activate dashboard
```

### Verify the installation

```bash
quarto --version   # Should show 1.6+ 
quarto check jupyter  # Verifies Jupyter integration works
```

> **Tip:** If you already have an existing micromamba/conda environment, you can just add Quarto to it:
> ```bash
> micromamba install -c conda-forge quarto -y
> ```

## What is a Quarto Dashboard?

Quarto Dashboards let you publish a group of related visualizations as a single page:

- **Rows and columns** defined with markdown headings (`##`, `###`) in Markdown cells
- **Cards** are created automatically from code cells
- **Tabsets, value boxes, and text cards** for rich layouts
- Works with Plotly, Matplotlib, Leaflet, and more
- Can be deployed as **static HTML** — no server needed



Quarto dashboards work **directly with Jupyter notebooks**. The mapping is:

::: {.callout-warning}
All layout cells (YAML header, row/column headings, page headings, `:::` cards) must be **Raw cells**, not Markdown cells. 

:::

## Dashboard Structure (in a notebook)

Your notebook should look like this:

```
┌─────────────────────────────────────────────┐
│ [Raw Cell]                                  │
│ ---                                         │
│ title: "My Dashboard"                       │
│ format: dashboard                           │
│   theme: cosmo                              │
│   scrolling: true                           │
│ execute:                                    │
│   enabled: true                             │
│ ---                                         │
├─────────────────────────────────────────────┤
│ [Raw Cell]                                  │
│ ## Row {height=60%}                         │
├─────────────────────────────────────────────┤
│ [Code Cell]                                 │
│ #| title: My Chart                          │
fig, ax = plt.subplots()                      │
├─────────────────────────────────────────────┤
│ [Raw Cell]                                  │
│ ## Row {height=40%}                         │
├─────────────────────────────────────────────┤
│ [Code Cell]                                 │
│ #| title: Chart A                           │
│ px.line(...)                                │
├─────────────────────────────────────────────┤
│ [Code Cell]                                 │
│ #| title: Chart B                           │
│ px.bar(...)                                 │
└─────────────────────────────────────────────┘
```

Two code cells after the same `## Row` heading become **side-by-side cards**.

> **Important:** All layout cells (YAML, `## Row`, `# Page`, `:::` cards) must be **Raw cells**. Only instruction/explanation text should be Markdown cells.

For plotly to display you will have to use: 

```
import plotly.io as pio
# Emit only the plotly mimetype — Quarto will render it with its own
# Plotly.js bundle and avoid the RequireJS "Mismatched anonymous define" clash
pio.renderers.default = "iframe"
```

## Key Layout Rules

| Syntax (in a Raw cell) | Effect |
|--------------------------|--------|
| `## Row` | Creates a new **row** |
| `## Row {height=70%}` | Row with **relative height** |
| `## Row {height=200px}` | Row with **defined height** |
| `### Column` | Creates a **column** inside a row |
| `## Row {.tabset}` | Row where each card becomes a **tab** |
| `# Page Name` | Creates a new **page** (level 1 heading) |
| `orientation: columns` | In YAML — switch default layout to **columns** |

> **Important:** In Jupyter, to create a **Raw cell**, use the cell type dropdown in the toolbar and select "Raw" (not "Markdown" or "Code"). All Quarto layout syntax goes in Raw cells.

---

## Now Let's Build It Step by Step

The cells below form a **complete working dashboard**. You can either:

1. **Copy these cells** into a new notebook called `my-dashboard.ipynb`, or
2. **Duplicate this notebook** and delete the instruction cells

---

## Step 1: Create the YAML Header (Raw Cell)

In your new notebook, change the **first cell** to a **Raw cell** and paste the following YAML:

```yaml
---
title: "Development Indicators"
author: "Your Name"
format:
  dashboard:
    theme: cosmo
    scrolling: true
execute:
  enabled: true

---
```

The `execute: enabled: true` line is **essential** — without it, Quarto won't run your code cells and no plots will appear.

> In JupyterLab: click the cell type dropdown (top toolbar) and select **Raw**.  
> In VS Code: click the cell language indicator and select **Raw**.

## Step 2: Add a Setup Cell

Add a **Code cell** that imports libraries and loads data. This cell runs but won't create a card (no `#| title`):

In [ ]:
import plotly.express as px
df = px.data.gapminder()

## Step 3: Add Your First Row

Add a **Raw cell** with the row heading, then a **Code cell** with the chart.

**Raw cell:**
```
## Row {height=60%}
```

**Code cell** (with `#| title:` to name the card):

In [ ]:
#| title: GDP and Life Expectancy
px.scatter(
    df, x="gdpPercap", y="lifeExp",
    animation_frame="year",
    animation_group="country",
    size="pop", color="continent",
    hover_name="country",
    log_x=True, size_max=45,
    range_x=[100, 100000],
    range_y=[25, 90]
)

## Step 4: Add a Second Row with Two Side-by-Side Cards

Two **Code cells** after the same `## Row` Raw cell become **side-by-side cards**.

**Raw cell:**
```
## Row {height=40%}
```

Then add **two Code cells** one after the other:

In [ ]:
#| title: Population
px.area(
    df, x="year", y="pop",
    color="continent",
    line_group="country"
)

In [ ]:
#| title: Life Expectancy
px.line(
    df, x="year", y="lifeExp",
    color="continent",
    line_group="country"
)

## Step 5: Preview Your Dashboard

Run in the terminal:

```bash
quarto preview my-dashboard.ipynb
```

You should see:

- A **navigation bar** with your title and author
- **Row 1** (60%): animated scatter plot
- **Row 2** (40%): two charts side by side

## Step 6: Add Tabsets

To turn cards into **tabs**, add `.tabset` to the row heading.

Replace the second row's Raw cell with:
```
## Row {height=40% .tabset}
```

Each Code cell after this heading becomes its own **tab** instead of a side-by-side card.

## Step 7: Add Value Boxes

Value boxes display key metrics prominently. Add a new row **above** your charts.

**Raw cell:**
```
## Row {height=20%}
```

Then add three Code cells with `#| content: valuebox`:

In [ ]:
#| content: valuebox
#| title: "Countries"
#| icon: globe
#| color: primary
dict(value = df["country"].nunique())

In [ ]:
#| content: valuebox
#| title: "Years Covered"
#| icon: calendar
dict(value = f'{df["year"].min()}–{df["year"].max()}')

In [ ]:
#| content: valuebox
#| title: "Continents"
#| icon: map
dict(value = df["continent"].nunique())

## Step 8: Add a Text Card

Use a fenced div in a **Raw cell** to add rich text as a card:

```
::: {.card title="About This Dashboard"}
This dashboard explores the **Gapminder** dataset,
tracking development indicators across countries
from 1952 to 2007.

Data source: [gapminder.org](http://www.gapminder.org/data/)
:::
```

Place this Raw cell inside a `## Row` alongside a Code cell to get a text card next to a chart.

> **Remember:** This must be a **Raw cell**, not Markdown — otherwise the `:::` syntax won't be processed by Quarto.

## Step 9: Add Multiple Pages

Use **level 1 headings** (`#`) in Raw cells to create pages with navigation tabs:

**Raw cell:**
```
# Overview
```

Then add rows and charts for page 1.

**Raw cell:**
```
# By Continent {orientation="columns"}
```

Then add columns and charts for page 2. Each `#` heading creates a separate **page tab** in the dashboard navigation.

## Step 10: Theming

Change the look with [Bootswatch themes](https://bootswatch.com/) in your Raw YAML cell:

```yaml
---
title: "Development Indicators"
format:
  dashboard:
    theme: cosmo
---
```

Available themes: `cosmo`, `flatly`, `darkly`, `lux`, `minty`, `pulse`, `sandstone`, `united`, ...

## Your Turn!


::: {.callout-note}

## Exercise

Build your own dashboard notebook with the plots from the msleep datasets with these **requirements**:

1. First cell is a **Raw cell** with `format: dashboard` and `execute: enabled: true` in the YAML
2. All layout headings (`## Row`, `# Page`, etc.) are **Raw cells**
3. At least **two rows** with different heights
4. At least **one row with two side-by-side cards** (two Code cells after one `## Row`)
6. At least **one tabset** (`.tabset`)
7. Add **value boxes** with key metrics

**Bonus challenges:**

- Add a **second page** with a different layout
- Add a **text card** using a `:::` fenced div in a Raw cell
- Try the `orientation: columns` layout
- Apply a **theme** to your dashboard

**Render your dashboard:**

```bash
quarto render my-dashboard.ipynb
```

:::

## Quarto Dashboard Cheat Sheet

| What you want | Syntax |
|---------------|--------|
| YAML header | **Raw cell** with `--- ... ---` (include `execute: enabled: true`) |
| New row | **Raw cell**: `## Row` or `## Row {height=60%}` |
| New column | **Raw cell**: `### Column` or `### Column {width=40%}` |
| Column-first layout | `orientation: columns` in YAML |
| Tabset | **Raw cell**: `## Row {.tabset}` |
| Card title | `#| title: My Title` in code cell |
| Value box | `#| content: valuebox` in code cell |
| Text card | **Raw cell**: `::: {.card title="..."} ... :::` |
| New page | **Raw cell**: `# Page Name` (level 1 heading) |
| Scrolling | `scrolling: true` in YAML |
| Theme | `theme: cosmo` in YAML |

## Further Reading

- Quarto Dashboards: [Official Documentation](https://quarto.org/docs/dashboards/)
- Quarto Dashboards: [Layout Guide](https://quarto.org/docs/dashboards/layout.html)
- Quarto Dashboards: [Data Display](https://quarto.org/docs/dashboards/data-display.html)
- Quarto Dashboards: [Theming](https://quarto.org/docs/dashboards/theming.html)
- Quarto Dashboards: [Examples Gallery](https://quarto.org/docs/gallery/#dashboards)
- Using Quarto with Jupyter: [Quarto & Jupyter](https://quarto.org/docs/tools/jupyter-lab.html)